# SG-VO — LoRA Online Adaptation Experiment
### ICRA Research Notebook

**Before running:** `Runtime → Change runtime type → T4 GPU`

Runs four conditions and compares them in a results table:

| # | Condition | Params updated |
|---|---|---|
| A | Offline VO (no adaptation) | 0 |
| B | Full online adaptation (paper) | 39M |
| C | LoRA-attention rank 8 | 21K (0.08%) |
| D | LoRA-attention + trigger 0.8 | 21K, ~50% of frames |

In [ ]:
# ── Cell 1: Check GPU ────────────────────────────────────────────────────────
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), 'GB')
else:
    print('⚠️  No GPU — go to Runtime → Change runtime type → T4 GPU')

In [ ]:
# ── Cell 2: Clone Repo and Install Dependencies ──────────────────────────────
import os
if not os.path.exists('/content/SG-VO'):
    !git clone https://github.com/AreebaAzizRajput/SG-VO.git /content/SG-VO
else:
    %cd /content/SG-VO
    !git pull
%cd /content/SG-VO
!pip install -q einops path imageio scikit-image tqdm gdown
print('✅ Done')

In [ ]:
# ── Cell 3: Download Pretrained Checkpoints ──────────────────────────────────
import os, glob, shutil
os.makedirs('checkpoints', exist_ok=True)

if not os.path.exists('checkpoints/exp_pose112_model_best.pth.tar'):
    print('Downloading pretrained models from Google Drive...')
    !gdown --folder 'https://drive.google.com/drive/folders/1twZsg2pxMLwcUUnFszBn1ryaEtuoEfs3' \
        -O checkpoints/ --quiet
    for f in glob.glob('checkpoints/**/*.pth.tar', recursive=True):
        dest = os.path.join('checkpoints', os.path.basename(f))
        if f != dest: shutil.move(f, dest)

!ls -lh checkpoints/*.pth.tar

In [ ]:
# ── Cell 4: Download Camera Intrinsics ───────────────────────────────────────
import os
os.makedirs('data/kitti_odom/sequences', exist_ok=True)

if not os.path.exists('data/kitti_odom/sequences/kitti_odom256_intrinsics/cam_09.txt'):
    print('Downloading camera intrinsics...')
    !gdown --folder 'https://drive.google.com/drive/folders/1n81QDHaG3lIxnxybl9I6knPHxPngTsD8' \
        -O data/kitti_odom/sequences/ --quiet

!ls data/kitti_odom/sequences/kitti_odom256_intrinsics/

In [ ]:
# ── Cell 5: Download KITTI Odometry Sequences ───────────────────────────────
# Strategy: download zip to /content (NOT /tmp which has limited tmpfs space),
# extract only seq 09 and 10, then delete the zip immediately to free 65 GB.
# If seq 10 extraction fails due to disk space, seq 09 alone is kept and used.
import os, shutil, glob

DATASET_DIR = 'data/kitti_odom/sequences'
os.makedirs(DATASET_DIR, exist_ok=True)
os.makedirs('vo_results_offline', exist_ok=True)
os.makedirs('vo_results_full', exist_ok=True)
os.makedirs('vo_results_lora', exist_ok=True)
os.makedirs('vo_results_lora_trigger', exist_ok=True)

# Download to /content (main disk) not /tmp (tmpfs, much smaller)
ZIP = '/content/kitti_odom_color.zip'

def seq_ok(seq):
    imgs = glob.glob(f'{DATASET_DIR}/{seq}/image_2/*.png')
    return len(imgs) > 100

if seq_ok('09'):
    print('✅ Seq 09 already present')
else:
    total, used, free = shutil.disk_usage('/')
    print(f'Disk: {free//1e9:.0f} GB free')
    if free < 68e9:
        print('⚠️  Less than 68 GB free — download may fail. Consider restarting with a fresh runtime.')

    if not os.path.exists(ZIP):
        print('Downloading KITTI (~65 GB, ~10-15 min)...')
        !wget -q --show-progress -c \
            'https://s3.eu-central-1.amazonaws.com/avg-kitti/data_odometry_color.zip' \
            -O {ZIP}

    # Extract directly to content — avoids double-copying via /tmp
    TMP_EXTRACT = '/content/kitti_raw'
    print('Extracting seq 09...')
    !unzip -q {ZIP} 'dataset/sequences/09/*' -d {TMP_EXTRACT} 2>/dev/null || true

    print('Extracting seq 10...')
    !unzip -q {ZIP} 'dataset/sequences/10/*' -d {TMP_EXTRACT} 2>/dev/null || true

    # Delete zip immediately to free 65 GB
    if os.path.exists(ZIP):
        os.remove(ZIP)
        print('Zip deleted — 65 GB freed')

    # Move extracted sequences to final location
    for seq in ['09', '10']:
        src  = f'{TMP_EXTRACT}/dataset/sequences/{seq}'
        dest = f'{DATASET_DIR}/{seq}'
        if os.path.exists(src) and not os.path.exists(dest):
            shutil.move(src, dest)
            n = len(glob.glob(f'{dest}/image_2/*.png'))
            print(f'Seq {seq}: {n} images moved ✅')
        elif not os.path.exists(src):
            print(f'Seq {seq}: extraction incomplete — skipping')

    if os.path.exists(TMP_EXTRACT):
        shutil.rmtree(TMP_EXTRACT, ignore_errors=True)

# Final check
SEQUENCES = [s for s in ['09','10'] if seq_ok(s)]
print(f'\nAvailable sequences: {SEQUENCES}')
for s in SEQUENCES:
    n = len(glob.glob(f'{DATASET_DIR}/{s}/image_2/*.png'))
    print(f'  Seq {s}: {n} images')

### Recovery Cell — only run this if Cell 5 crashed mid-extraction

If the notebook failed during extraction (disk full error), run this cell to recover what was saved and free disk space before continuing.

In [ ]:
# ── Cell 5b: Recovery (run only if Cell 5 crashed) ──────────────────────────
import os, shutil, glob

DATASET_DIR = 'data/kitti_odom/sequences'
TMP_EXTRACT = '/content/kitti_raw'
ZIP         = '/content/kitti_odom_color.zip'

# Step 1: delete zip to free 65 GB
for path in [ZIP, '/tmp/data_odometry_color.zip']:
    if os.path.exists(path):
        sz = os.path.getsize(path)/1e9
        os.remove(path)
        print(f'Deleted {path} ({sz:.1f} GB freed)')

# Step 2: move whatever sequences were extracted
for seq in ['09', '10']:
    for src_root in [TMP_EXTRACT, '/tmp/kitti_raw']:
        src = f'{src_root}/dataset/sequences/{seq}'
        dest = f'{DATASET_DIR}/{seq}'
        if os.path.exists(src) and not os.path.exists(dest):
            shutil.move(src, dest)
            n = len(glob.glob(f'{dest}/image_2/*.png'))
            print(f'Seq {seq}: moved {n} images ✅')

# Step 3: clean up temp dirs
for tmp in [TMP_EXTRACT, '/tmp/kitti_raw']:
    if os.path.exists(tmp):
        shutil.rmtree(tmp, ignore_errors=True)
        print(f'Cleaned {tmp}')

# Step 4: report
total, used, free = shutil.disk_usage('/')
print(f'\nDisk now: {free//1e9:.0f} GB free')
SEQUENCES = [s for s in ['09','10']
             if len(glob.glob(f'{DATASET_DIR}/{s}/image_2/*.png')) > 100]
print(f'Available sequences: {SEQUENCES}')
for s in SEQUENCES:
    n = len(glob.glob(f'{DATASET_DIR}/{s}/image_2/*.png'))
    print(f'  Seq {s}: {n} images')

In [ ]:
# ── Cell 6: Install cam.txt Intrinsics ───────────────────────────────────────
import os, shutil, glob

INTR_DIR    = 'data/kitti_odom/sequences/kitti_odom256_intrinsics'
DATASET_DIR = 'data/kitti_odom/sequences'
SEQUENCES   = [s for s in ['09','10']
               if len(glob.glob(f'{DATASET_DIR}/{s}/image_2/*.png')) > 100]

for seq in SEQUENCES:
    src  = f'{INTR_DIR}/cam_{seq}.txt'
    dest = f'{DATASET_DIR}/{seq}/image_2/cam.txt'
    if os.path.exists(src):
        shutil.copy(src, dest)
        print(f'✅ cam.txt → seq {seq}')
    else:
        print(f'⚠️  Missing intrinsics for seq {seq}: {src}')

In [ ]:
# ── Cell 7: LoRA Sanity Check ────────────────────────────────────────────────
# PoseResNet ALWAYS uses ResNet-18 regardless of --resnet-layers flag.
# (--resnet-layers only controls DispResNet, the depth network.)
# Using num_layers=50 here caused a channel mismatch: ResNet-50 outputs
# 2048-channel features but crossAttention expects 512.
import sys, torch
sys.path.insert(0, '/content/SG-VO')
from lora import inject_lora_pose_net, freeze_all, count_lora_params
from models import PoseResNet

pose_net = PoseResNet(num_layers=18, pretrained=False)  # always 18 for pose
n_total  = sum(p.numel() for p in pose_net.parameters())
freeze_all(pose_net)
inject_lora_pose_net(pose_net, rank=8, alpha=1.0, targets='attention')
n_lora = count_lora_params(pose_net)
print(f'LoRA params: {n_lora:,} / {n_total:,}  ({100*n_lora/n_total:.3f}%)')
print(f'Expected:    21,504 / 25,735,766  (0.084%)')

with torch.no_grad():
    out = pose_net(torch.randn(1, 36, 256, 832), torch.randn(1, 36, 256, 832))
print(f'Forward pass OK — output shape: {out.shape}')
print('✅ LoRA ready')

## Condition A — Offline VO (baseline, no adaptation)

In [ ]:
# ── Cell 8: Offline VO ───────────────────────────────────────────────────────
import glob
DATASET_DIR = 'data/kitti_odom/sequences/'
POSE_NET    = 'checkpoints/exp_pose112_model_best.pth.tar'
SEQUENCES   = [s for s in ['09','10']
               if len(glob.glob(f'data/kitti_odom/sequences/{s}/image_2/*.png')) > 100]
print(f'Running on sequences: {SEQUENCES}')

for seq in SEQUENCES:
    print(f'\n=== Offline VO — Seq {seq} ===')
    !python test_vo.py \
        --img-height 256 --img-width 832 \
        --sequence {seq} \
        --pretrained-posenet {POSE_NET} \
        --dataset-dir {DATASET_DIR} \
        --output-dir vo_results_offline/

print('\n=== Offline results ===')
!python kitti_eval/eval_odom.py --result=vo_results_offline/ --align=7dof

## Condition B — Full Online Adaptation (paper baseline)

In [ ]:
# ── Cell 9: Full Online VO ───────────────────────────────────────────────────
import glob
DATASET_DIR = 'data/kitti_odom/sequences/'
POSE_NET    = 'checkpoints/exp_pose112_model_best.pth.tar'
DISP_NET    = 'checkpoints/dispnet112_model_best.pth.tar'
SEQUENCES   = [s for s in ['09','10']
               if len(glob.glob(f'data/kitti_odom/sequences/{s}/image_2/*.png')) > 100]

for seq in SEQUENCES:
    print(f'\n=== Full Online VO — Seq {seq} ===')
    !python test_vo_online.py \
        -p 1 -s 0.1 -c 0.5 \
        --img-height 256 --img-width 832 \
        --sequence {seq} \
        --pretrained-posenet {POSE_NET} \
        --pretrained-disp    {DISP_NET} \
        --dataset-dir {DATASET_DIR} \
        --output-dir  vo_results_full/ \
        --epochs 2 --lr 1e-4 \
        --sequence-length 3 \
        --resnet-layers 50 \
        --select-best 1 \
        --with-mask 1 --with-auto-mask 1 \
        --padding-mode border

print('\n=== Full online results ===')
!python kitti_eval/eval_odom.py --result=vo_results_full/ --align=7dof

## Condition C — LoRA-Attention Adaptation (new contribution)

In [ ]:
# ── Cell 10: LoRA Online VO ──────────────────────────────────────────────────
import glob
DATASET_DIR = 'data/kitti_odom/sequences/'
POSE_NET    = 'checkpoints/exp_pose112_model_best.pth.tar'
DISP_NET    = 'checkpoints/dispnet112_model_best.pth.tar'
SEQUENCES   = [s for s in ['09','10']
               if len(glob.glob(f'data/kitti_odom/sequences/{s}/image_2/*.png')) > 100]

for seq in SEQUENCES:
    print(f'\n=== LoRA Online VO — Seq {seq} ===')
    !python test_vo_online.py \
        -p 1 -s 0.1 -c 0.5 \
        --img-height 256 --img-width 832 \
        --sequence {seq} \
        --pretrained-posenet {POSE_NET} \
        --pretrained-disp    {DISP_NET} \
        --dataset-dir {DATASET_DIR} \
        --output-dir  vo_results_lora/ \
        --epochs 2 --lr 1e-4 \
        --sequence-length 3 \
        --resnet-layers 50 \
        --select-best 1 \
        --with-mask 1 --with-auto-mask 1 \
        --padding-mode border \
        --lora-rank 8 \
        --lora-targets attention

print('\n=== LoRA results ===')
!python kitti_eval/eval_odom.py --result=vo_results_lora/ --align=7dof

## Condition D — LoRA + Trigger (efficiency variant)

In [ ]:
# ── Cell 11: LoRA + Trigger ──────────────────────────────────────────────────
import glob
DATASET_DIR = 'data/kitti_odom/sequences/'
POSE_NET    = 'checkpoints/exp_pose112_model_best.pth.tar'
DISP_NET    = 'checkpoints/dispnet112_model_best.pth.tar'
SEQUENCES   = [s for s in ['09','10']
               if len(glob.glob(f'data/kitti_odom/sequences/{s}/image_2/*.png')) > 100]

for seq in SEQUENCES:
    print(f'\n=== LoRA + Trigger — Seq {seq} ===')
    !python test_vo_online.py \
        -p 1 -s 0.1 -c 0.5 \
        --img-height 256 --img-width 832 \
        --sequence {seq} \
        --pretrained-posenet {POSE_NET} \
        --pretrained-disp    {DISP_NET} \
        --dataset-dir {DATASET_DIR} \
        --output-dir  vo_results_lora_trigger/ \
        --epochs 2 --lr 1e-4 \
        --sequence-length 3 \
        --resnet-layers 50 \
        --select-best 1 \
        --with-mask 1 --with-auto-mask 1 \
        --padding-mode border \
        --lora-rank 8 \
        --lora-targets attention \
        --adapt-threshold 0.8

print('\n=== LoRA + Trigger results ===')
!python kitti_eval/eval_odom.py --result=vo_results_lora_trigger/ --align=7dof

## Results Comparison Table

In [ ]:
# ── Cell 12: Comparison Table ────────────────────────────────────────────────
import subprocess, re, os

PAPER = {
    'offline': {'09': (7.08, 2.48), '10': (8.72, 3.11)},
    'online':  {'09': (5.21, 1.93), '10': (6.74, 2.57)},
}

def parse_eval(result_dir):
    out = subprocess.run(
        ['python', 'kitti_eval/eval_odom.py',
         f'--result={result_dir}', '--align=7dof'],
        capture_output=True, text=True).stdout
    results = {}
    for seq in ['09', '10']:
        m = re.search(rf'{seq}.*?([0-9]+\.[0-9]+).*?([0-9]+\.[0-9]+)', out)
        results[seq] = (float(m.group(1)), float(m.group(2))) if m else (None, None)
    return results

experiments = [
    ('A  Offline VO',         'vo_results_offline'),
    ('B  Full online (paper)', 'vo_results_full'),
    ('C  LoRA-attn rank 8',    'vo_results_lora'),
    ('D  LoRA + trigger 0.8',  'vo_results_lora_trigger'),
]

W = 28
print(f"{'Condition':<{W}} | {'Seq 09':^19} | {'Seq 10':^19}")
print(f"{'':< {W}} | {'t_err%':>9} {'r°/100m':>8} | {'t_err%':>9} {'r°/100m':>8}")
print('-' * 72)
for label, key in [('  Paper offline', 'offline'), ('  Paper online', 'online')]:
    r09, r10 = PAPER[key]['09'], PAPER[key]['10']
    print(f"{label:<{W}} | {r09[0]:9.2f} {r09[1]:8.2f} | {r10[0]:9.2f} {r10[1]:8.2f}")
print('-' * 72)
for name, d in experiments:
    if not os.path.exists(d):
        continue
    r = parse_eval(d)
    def fmt(v): return f'{v:9.2f}' if v else '      N/A'
    t09, r09 = r.get('09', (None, None))
    t10, r10 = r.get('10', (None, None))
    print(f"{name:<{W}} | {fmt(t09)} {fmt(r09)} | {fmt(t10)} {fmt(r10)}")

## Trajectory Visualization

In [ ]:
# ── Cell 13: Trajectory Plots ────────────────────────────────────────────────
exec(open('scripts/colab_eval_viz.py').read())

In [ ]:
# ── Cell 14: Download All Results ────────────────────────────────────────────
import zipfile, os
from google.colab import files

with zipfile.ZipFile('/content/sgvo_lora_results.zip', 'w') as z:
    for d in ['vo_results_offline', 'vo_results_full',
              'vo_results_lora', 'vo_results_lora_trigger']:
        if not os.path.exists(d): continue
        for f in os.listdir(d):
            z.write(os.path.join(d, f))

files.download('/content/sgvo_lora_results.zip')